In [1]:
import datetime as dt

import luigi
import numpy as np
import pandas as pd
import plotly.express as px
from moexalgo import CandlePeriod

from config import LUIGI_OUTPUT_DIR
from src.auction_model import AuctionModelSample, TargetType
from src.instruments import symbol_to_series
from src.qr import get_mos_dates_range
from src.tasks import YieldCandles, BondMetricsAtVWAP, BondType, Board

In [5]:
dates = get_mos_dates_range(dt.date(2025, 3, 10), 2)

samples = [AuctionModelSample(date=date, out_dir=LUIGI_OUTPUT_DIR, time_of_day=dt.timedelta(hours=18), target_type=TargetType.FIRST) for date in dates]
tasks = samples

if not all(t.complete() for t in tasks):
    luigi.build(tasks, local_scheduler=True)

dfs = []

for sample, date in zip(samples, dates):
    yields = YieldCandles(date=date, out_dir=LUIGI_OUTPUT_DIR, bond_type=BondType.FIX, board=Board.TQOB, period=CandlePeriod.ONE_HOUR)
    metrics_df = yields.clone(BondMetricsAtVWAP).read_output()
    yields_df = yields.read_output()


    df = yields_df[yields_df['end'].dt.hour == 18][['weighted_avg']].join(metrics_df[['duration']])\
            .join(sample.read_output().set_index('secid')[['issue_size_ts', 'issue_size_mln', 'remaining_size_mln', 'placement_end_date']].round()).query('duration > 3').reset_index()
    df['date'] = df['begin'].dt.normalize()
    df['series'] = symbol_to_series(df['secid']) if date == dates[-1] else np.nan
    df['is_star'] = (df['issue_size_ts'].dt.date == date).astype(float)
    dfs.append(df)

df = pd.concat(dfs)

fig = px.scatter(df,
                 x='duration',
                 y='weighted_avg',
                 color='begin',
                 # hover_name='secid',
                 hover_data=['remaining_size_mln', 'issue_size_mln', 'placement_end_date'],
                 text='series',
                 symbol='is_star',
                 symbol_map={1.: 'star', 0.: 'circle'})

fig.update_traces(textposition='top center',
                  textfont_size=10,
                  textfont_color='rgba(255,255,255,0.4)')